In [ ]:
import pandas as panda
from pandas import DataFrame
import matplotlib.pyplot as plot
import numpy as numpy
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# creating a dataframe from the nutrients dataset
df = panda.read_csv('dataset/e1_nutrients.csv')

#split the large dataset into training and test datasets
test_data_df, training_data_df = train_test_split(df, test_size=0.2, random_state=42)

filtered_training_df = training_data_df.copy()
original_df_copy = df.copy()

def filter_outliers(df, column):
    Q3 = df[column].quantile(0.75)
    Q1 = df[column].quantile(0.25)
    IQR = Q3 - Q1
    higher_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR
    # outlier_mask = (df[column] < lower_bound) | (df[column] > higher_bound)
    # df.loc[outlier_mask, column] = None
    df[column] = df[column].where((df[column] >= lower_bound) & (df[column] <= higher_bound), None)
    return df

for depth in filtered_training_df['Depth'].unique():
    mask = filtered_training_df['Depth'] == depth
    depth_set = filtered_training_df.loc[mask].copy()
    for chemical in ['NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']:
        depth_set = filter_outliers(depth_set, chemical)
    filtered_training_df.loc[mask] = depth_set
cleaned_trained_df = filtered_training_df.dropna()
# # we are passing the training dataset instead of the test data df as we want to keep test_data_df as a separate dataset to evaluate our model on after we have trained it on the training dataset
# # get all values witihin the column depth from the dataframe
# depth_values = filtered_training_df['Depth']
    
# # because values in the depth column are repeated, i.e. numbers 0-60 are repeated multiple times, filter out the repeated values and get only the unique values, using .unique()
# for depth in depth_values.unique():
#     # because we currently have all values for the depth column, we want to narrow our focus to only values for the current depth 
#     # so we copy the dataframe and filter it to only include the rows where the depth value is equal to the current depth value we are iterating over
#     current_depth_values = filtered_training_df.loc[depth_values == depth].copy()
        
#     # we want to get values for each chemical given a depth val
#     # so we create a list of chemicals fromt he nutrients dataset we want to loop through, excluding NITRITE
#     # we dont want to filter NITRITE since this is the chemical we want to predict
#     # including outliers is helpful because it gives us a better idea of the range of values NITRITE can take, which is important for our prediction task
#     chemical_list = ['NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']
        
#     # iterate through the chemical list 
#     for chemical in chemical_list:
            
#     # for each chemical, we want to get the values for that chemical at the current depth value we are iterating over
#         depth_chemical_values = current_depth_values[chemical]
            
#         # to remove the outliers, we have to caluclate the interquartile range IQR by finding the upper quartile Q3 and lower quartile Q1
#         # numpy has a built-in function quantile-- the value 0.75 gives the upper range and 0.25 gives the lower range
#         Q3 = depth_chemical_values.quantile(0.75)
#         Q1 = depth_chemical_values.quantile(0.25)
            
#         # subtracting the bigger quartile from the smaller quartile
#         IQR = Q3 - Q1
            
#         # we have to calculate higher bound and lower bound using the quartiles and IQR
#         higher_bound = Q3 + 1.5 * IQR
#         lower_bound = Q1 - 1.5 * IQR
            
#         for index, value in enumerate(depth_chemical_values):
#             if value > higher_bound or value < lower_bound:
#                 depth_chemical_values.iloc[index] = None
            
#             current_depth_values[chemical] = depth_chemical_values
#         filtered_training_df.loc[depth_values == depth] = current_depth_values
# cleaned_trained_df = filtered_training_df.dropna()
    
# scaling the data 
x_scaler = RobustScaler()
y_scaler = RobustScaler()
x_train = x_scaler.fit_transform(cleaned_trained_df.drop(columns=['NITRITE'])) # all but NITRITE
y_train = y_scaler.fit_transform(cleaned_trained_df[['NITRITE']]).ravel() # target (NITRITE)

# scaling test data
x_test = x_scaler.transform(test_data_df.drop(columns=['NITRITE']))
y_test = y_scaler.transform(test_data_df[['NITRITE']]).ravel()

filtered_df = filtered_training_df
scaled_df = cleaned_trained_df.copy().astype(float)
scale_columns = scaled_df.columns != "NITRITE"
scaled_df.loc[:, scale_columns] = x_train

#--------------
# scatter graphs to visualise the original data against the cleaned and scaled data
chemical_list = ['NITRITE', 'NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']
    
fig, axes = plot.subplots(2, 3, figsize=(16, 8), sharex=False)
axes = axes.flatten()

for ax, chemical in zip(axes, chemical_list):
    ax.scatter(
        original_df_copy['Depth'] +2.0,
        original_df_copy[chemical],
        color='#d4334e',
        label='Original'
            
    )
    ax.scatter(
        cleaned_trained_df['Depth'],
        cleaned_trained_df[chemical],
        label='Cleaned',
        color='#32a852'
            
    )
    
    ax2 = ax.twiny()
    ax2.set_xlim(-1.1, 1.1)
        
    ax2.scatter(
        scaled_df['Depth']+0.01,
        scaled_df[chemical],
        label='Scaled',
        color='#3252a8'
    )

    ax.set_title(f'Depth vs {chemical}')
    ax.set_xlabel('Depth')
    ax.set_ylabel(chemical)
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    
for ax in axes[len(chemical_list):]:
    ax.remove()
        
plot.tight_layout()
plot.show()
    


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
   
linear_regression_model = LinearRegression()
random_forest_model = RandomForestRegressor(random_state=42)
neural_network_model = MLPRegressor(random_state=42, max_iter=2500)

# apply tuning to the neural network model to improve its performance
tuned_nn_model = MLPRegressor(random_state=42, max_iter=2500, hidden_layer_sizes=(100, 50), alpha=0.001)

"""Linear regression."""
# linear regression fit to the training data and making predictions on both the training and test data
linear = linear_regression_model.fit(x_train, y_train)
linear_train_prediction = linear.predict(x_train)
linear_test_prediction = linear.predict(x_test)
      
# linear training score
train_linear_mse = mean_squared_error(y_train, linear_train_prediction)
test_linear_mse = mean_squared_error(y_test, linear_test_prediction)
   
"""Random forest regression."""
# fitting the random forest model to the training data and making predictions on both the training and test data
rf = random_forest_model.fit(x_train, y_train)
random_forest_train_prediction = rf.predict(x_train)
random_forest_test_prediction = rf.predict(x_test)

# random forest training score
random_forest_train_mse = mean_squared_error(y_train, random_forest_train_prediction)
random_forest_test_mse = mean_squared_error(y_test, random_forest_test_prediction)

"""Neural Network regression (NOT TUNED)."""
# fitting the neural network model to the training data and making predictions on both the training and test data
neural = neural_network_model.fit(x_train, y_train)
neural_network_train_prediction = neural.predict(x_train)
neural_network_test_prediction = neural.predict(x_test)

# training score for neural network
neural_network_train_mse = mean_squared_error(y_train, neural_network_train_prediction)
neural_network_test_mse = mean_squared_error(y_test, neural_network_test_prediction)

"""TUNED neural network regression."""
tuned_neural = tuned_nn_model.fit(x_train, y_train)
tuned_nn_train_prediction = tuned_neural.predict(x_train)
tuned_nn_test_prediction = tuned_neural.predict(x_test)

# training score for tuned neural network
tuned_nn_train_mse = mean_squared_error(y_train, tuned_nn_train_prediction)
tuned_nn_test_mse = mean_squared_error(y_test, tuned_nn_test_prediction)

# cross validation for all models of regression
linear_cross_val = cross_val_score(linear_regression_model, x_train, y_train, cv=5, scoring='neg_mean_squared_error')
rf_cross_val = cross_val_score(random_forest_model, x_train, y_train, cv=5, scoring='neg_mean_squared_error')
nn_cross_val = cross_val_score(neural_network_model, x_train, y_train, cv=5, scoring='neg_mean_squared_error')
tuned_nn_cross_val = cross_val_score(tuned_nn_model, x_train, y_train, cv=5, scoring='neg_mean_squared_error')

# create a dictionary to organise the values to their respective model names for the panda DataFrame
scores_data = {
    'model name': ["Linear Regression", "Random Forest Regression", "Neural Network Regression", "Tuned Neural Network Regression"],
    'cross_value_mean': [linear_cross_val.mean(), rf_cross_val.mean(), nn_cross_val.mean(), tuned_nn_cross_val.mean()],
    'train_mse': [train_linear_mse, random_forest_train_mse, neural_network_train_mse, tuned_nn_train_mse],
    'test_mse': [test_linear_mse, random_forest_test_mse, neural_network_test_mse, tuned_nn_test_mse],
    'cross_value_std': [linear_cross_val.std(), rf_cross_val.std(), nn_cross_val.std(), tuned_nn_cross_val.std()]
}

panda.DataFrame(scores_data).sort_values(by='test_mse')

In [ ]:
panda.DataFrame(scores_data).sort_values(by='test_mse')